# Milestone 10 - One agent or a team?

The help center from milestones 9a to 9d answers with one agent. Someone suggests a
**team**: a supervisor, researchers that each read one part of the help center, one
writer, and a checker. It sounds better. Is it?

This notebook runs that experiment for real, the way the lesson **One agent or a
team: the experiment** and the plan **A worked example: one agent or a team?** in
`docs/EXPERIMENT_PLAN.md` describe it:

1. Write the plan and the decision rule, before anything runs.
2. Build the baseline: one agent with a search tool.
3. Build the team, one piece at a time.
4. Run all three designs (one agent, the team, and the team without its checker) on the
   same 20 questions, 3 times each, traced in MLflow.
5. Read the results: pass rate, passed every time, dollars per success, hand-off facts
   kept, tokens per agent and failure labels, side by side.
6. Apply the rule, and only the rule.

There are five small exercises on the way. Each one has tests underneath it.

**Do you need Ollama?** No. Every model reply was recorded on a MacBook Pro running
`qwen3:14b` in Ollama, and is replayed here, so the notebook gives the same numbers on
every computer. (`gemma3:12b`, the model in 9a to 9d, can't call tools in Ollama, and
agents need tools.) To run it yourself, `ollama pull qwen3:14b`, set `LIVE = True` below
and allow about 20 minutes. Your numbers will differ a little: that is what repeat runs
are for.

In [ ]:
import json
import operator
import os
from typing import Annotated, TypedDict

import ipytest
import mlflow
import pandas as pd
import pytest
from langchain.agents import create_agent
from langchain.agents.middleware import ModelCallLimitMiddleware
from langchain_core.tools import tool
from langgraph.graph import END, START, StateGraph
from langgraph.types import Send
from mlflow.entities import AssessmentSource, Feedback, SpanType
from mlflow.genai.scorers import scorer
from mlflow.tracing.constant import SpanAttributeKey

from drill.rag_lab import team as T

ipytest.autoconfig(clean="test_*")

LIVE = False   # True: every model call goes to your Ollama (qwen3:14b), about 20 minutes

lab = T.Lab(live=LIVE)
URI = T.use_mlflow("m10-one-agent-or-a-team")
mlflow.langchain.autolog()                                # every LangChain and LangGraph step becomes a span
os.environ["MLFLOW_GENAI_EVAL_MAX_WORKERS"] = "1"         # one question at a time, so timings are fair on a laptop
os.environ["MLFLOW_GENAI_EVAL_SKIP_TRACE_VALIDATION"] = "true"   # skip evaluate's trial call: our functions are traced already

print("Live: replies come from Ollama." if lab.live else "Offline: model replies are replayed from the recording.")
print(len(T.QUESTIONS), "questions,", len(lab.recorded), "recorded model replies")

## 1. Write the plan first

The plan is written before any result exists, so a lucky run can't move the goalposts.
Section 6 applies the decision rule exactly as written here.

Two numbers in it need a word:

- **Passed every time.** Agents give different answers on different tries. A question
  counts as passed every time only if all 3 tries passed. It's the "Reliable every time
  (pass hat k)" idea.
- **Dollars per success.** The whole bill, failed runs included, divided by the runs
  that passed. The model here runs on your laptop for free, so the notebook prices its
  token counts at example Azure AI Foundry prices: $2 per million tokens sent and $10
  per million tokens received (from `docs/research/LLM_COST_AND_SIZING.md`).

In [ ]:
PLAN = {
    "question": "Does a team answer help-center questions more reliably than one agent, at a cost we accept?",
    "change": "the design: one agent, or a supervisor with researchers, one writer and a checker",
    "fixed": f"model {T.MODEL} at temperature {T.TEMPERATURE}, the help center and its search, the scorer, the price table",
    "baseline": "one agent: the design that runs today",
    "variants": "one-agent, team, and team-no-checker (the ablation: the team with its checker taken out)",
    "eval_set": "20 questions: 7 one-article, 9 two-article, 4 the help center can't answer",
    "metric": "passed every time: the share of questions passed on all 3 tries, the answer checked by code",
    "also_report": "pass rate, dollars per success, model seconds, hand-off facts kept, tokens per agent, failure labels",
    "guardrails": "a run that hits a step limit, or that the checker sends to a person, counts as failed",
    "runs": "3 per question for each design: 60 runs per design",
    "decision_rule": "switch to the team only if its passed every time beats one agent's by 0.15 or more "
                     "AND its dollars per success is at most 0.05",
    "no_result": "keep one agent; label the failures and fix the biggest group first",
}
RULE = {"min_gain": 0.15, "max_cost_per_success": 0.05}
RUNS = 3

for field, value in PLAN.items():
    print(f"{field:14} {value}")

**How big a gap can 20 questions show?** Each question is 5 points of "passed every
time" (100 divided by 20). So a gap of one or two questions is noise. That's why the
rule asks for 0.15, three questions, and why section 5 looks at *which* questions
changed, not only the totals.

## 2. The questions, and how an answer is checked

The 20 questions come from the help-center eval set. Seven ask about one article, nine
ask two things at once (customers often do), and four ask something the help center
doesn't cover, where the right answer is to say so.

In [ ]:
for q in T.QUESTIONS:
    print(f"{q['id']:12} {q['group']:13} {q['question']}")

**The check is code, not the agent's word.** `end_state` passes an answer only if:

- every expected fact is in it (or, for a question the help center can't answer, it
  says it doesn't know),
- it contains no email address that the help center doesn't have (a made-up contact
  sends a customer nowhere),
- the run didn't stop at a step limit.

It's an MLflow `@scorer`. MLflow passes it the question (`inputs`), the design's reply
(`outputs`) and the expected facts (`expectations`), and it returns a `Feedback` that is
saved on the trace.

In [ ]:
DATA = [{"inputs": {"qid": q["id"], "question": q["question"]},
         "expectations": {"facts": q["facts"]}} for q in T.QUESTIONS]


@scorer
def end_state(inputs, outputs, expectations):
    answer = outputs["answer"]
    if outputs["stopped"]:
        return Feedback(value=False, rationale="stopped at a step limit or sent to a person")
    made_up = T.made_up_contacts(answer)
    if made_up:
        return Feedback(value=False, rationale=f"made-up contact: {made_up}")
    ok = T.score(answer, {"facts": expectations["facts"]})
    return Feedback(value=ok, rationale="all facts present" if ok else "a fact is missing, or a wrong decline")


print(end_state(inputs=DATA[0]["inputs"], expectations=DATA[0]["expectations"],
                outputs={"answer": "We retry it 3 times over 7 days.", "stopped": False}))

## 3. The baseline: one agent

One agent, made with LangChain's `create_agent`, with one tool: search the whole help
center. It decides when to search, how often, and when to answer. A **step limit**
(`ModelCallLimitMiddleware`) stops it after 6 model calls, so a confused agent can't loop
for ever.

Every design gets a **named span** of type `AGENT`. That's the labelled box in the
trace that lets MLflow show tokens per agent later.

In [ ]:
@tool
def search_help_center(query: str) -> str:
    """Search the shop's whole help center. Returns the 3 best-matching articles."""
    return T.search(query, k=3)


SOLO_PROMPT = f"""You answer customer questions for an online shop, using only its help center.
Use the search_help_center tool. If a part of the question is still unanswered, search again with other words.
Then answer in under 80 words. Include the exact numbers, timeframes, menu paths and email addresses from the articles.
If the help center doesn't answer the question, reply exactly: {T.DECLINE}"""

solo = create_agent(lab.model("agent"), tools=[search_help_center], system_prompt=SOLO_PROMPT,
                    middleware=[ModelCallLimitMiddleware(run_limit=6, exit_behavior="end")])


def one_agent(question):
    with mlflow.start_span(name="agent", span_type=SpanType.AGENT) as span:
        span.set_inputs({"question": question})
        result = solo.invoke({"messages": [{"role": "user", "content": question}]})
        answer = result["messages"][-1].content
        stopped = answer.startswith("Model call limits exceeded")
        span.set_outputs({"answer": answer})
    return {"answer": answer, "stopped": stopped}

Try it on one two-part question, and look at the trace it made. Each line is a span:
the indent shows what ran inside what.

In [ ]:
def show_trace(trace_id):
    trace = mlflow.get_trace(trace_id)
    by_id = {s.span_id: s for s in trace.data.spans}
    shown = ("AGENT", "TOOL", "CHAT_MODEL")       # skip the many small CHAIN steps
    for s in trace.data.spans:
        if s.span_type not in shown and s.parent_id is not None:
            continue
        depth, parent = 0, by_id.get(s.parent_id)
        while parent is not None:                 # indent once for each box this span sits in
            depth += parent.span_type in shown or parent.parent_id is None
            parent = by_id.get(parent.parent_id)
        usage = s.get_attribute(SpanAttributeKey.CHAT_USAGE)
        tokens = f"  {usage['input_tokens']} in, {usage['output_tokens']} out" if usage else ""
        print("  " * depth + f"{s.span_type:10} {s.name}{tokens}")


QUESTION = T.BY_ID["two-01"]
with lab.request("demo", QUESTION["id"]):
    with mlflow.start_span(name="demo") as root:
        reply = one_agent(QUESTION["question"])
mlflow.flush_trace_async_logging()
print(QUESTION["question"], "\n->", reply["answer"], "\n")
show_trace(root.trace_id)

## 4. The team, one piece at a time

The team follows the module's lessons:

| Piece | What it does | Lesson |
|---|---|---|
| **researchers** | one per part of the help center, all at the same time, read-only | Fan out, then gather |
| **supervisor** | decides which parts to search; the researchers are its tool | Comparing agent designs |
| **writer** | the only one who writes the answer | Split by what each agent needs to know |
| **checker** | a fresh pair of eyes with a written checklist, at most 3 rounds | A checker with a round limit |

### 4a. Researchers: fan out, then gather

The help center has four sections: billing, shipping, returns and security. A
researcher searches one section and reports the facts it found. Researchers only
read, so they can safely run at the same time.

LangGraph's `Send` starts one researcher per section. They all add their report to
the same key, `reports`. First, see what happens **without** a reducer (the gotcha
card "Two workers, one key"):

In [ ]:
class NoReducer(TypedDict):
    sections: list[str]
    reports: list                       # a plain key: one value per step


def fan_out(state):
    return [Send("researcher", {"section": s}) for s in state["sections"]]


def fake_researcher(job):
    return {"reports": [job["section"] + ": nothing yet"]}


g = StateGraph(NoReducer)
g.add_node("researcher", fake_researcher)
g.add_conditional_edges(START, fan_out, ["researcher"])
g.add_edge("researcher", END)
try:
    g.compile().invoke({"sections": ["billing", "returns"], "reports": []})
except Exception as error:
    print(type(error).__name__, "-", str(error).splitlines()[0])

Two researchers wrote to one plain key in the same step, and LangGraph refused. A
**reducer** is the rule for combining their updates. `operator.add` joins lists, so
each report is added to the list.

The real researcher reads the 3 best articles in its section and copies out the facts
that help, **word for word**, as short phrases. Copying, not rewording, is what lets
the hand-off check in section 5 see what got lost.

In [ ]:
RESEARCH_PROMPT = """You are the {section} researcher for an online shop's help center.
Read the articles below and pick out every fact that helps answer the customer's question.
Copy each fact word for word from an article, as a short phrase of at most 10 words.
Reply with JSON only, like {{"facts": ["Express shipping costs $15"]}}. If nothing helps, reply {{"facts": []}}.

Question: {question}

{articles}"""


class Research(TypedDict):
    question: str
    sections: list[str]
    reports: Annotated[list, operator.add]     # the reducer: every researcher's report is added


def fan_out(state):
    return [Send("researcher", {"question": state["question"], "section": s}) for s in state["sections"]]


def researcher(job):
    section = job["section"]
    with mlflow.start_span(name="researcher", span_type=SpanType.AGENT) as span:
        span.set_inputs(job)
        articles = T.search(job["question"], k=3, section=section)
        prompt = RESEARCH_PROMPT.format(section=section, question=job["question"], articles=articles)
        reply = lab.model(f"researcher_{section}").invoke(prompt).content
        report = {"section": section, "facts": T.read_json(reply, "facts")}
        span.set_outputs(report)
    return {"reports": [report]}


g = StateGraph(Research)
g.add_node("researcher", researcher)
g.add_conditional_edges(START, fan_out, ["researcher"])
g.add_edge("researcher", END)
research_graph = g.compile()

with lab.request("demo", QUESTION["id"]):
    found = research_graph.invoke({"question": QUESTION["question"], "sections": ["returns", "billing"], "reports": []})
for report in found["reports"]:
    print(report["section"], report["facts"])

### 4b. The supervisor: an agent whose tool is the researchers

The supervisor is made with `create_agent`, like the one agent, but its only tool is
`research`, which runs the fan-out above. It picks the sections; it never answers the
customer. (The older `langgraph-supervisor` package is archived. This is the current
way to build a supervisor.)

The tool returns two things: text for the supervisor to read, and the reports
themselves as an **artifact**. The team passes the artifact on to the writer word for
word, so the supervisor can't play telephone with the facts.

In [ ]:
@tool(response_format="content_and_artifact")
def research(question: str, sections: list[str]):
    """Send researchers to sections of the help center, all at once. sections: any of billing, shipping, returns, security."""
    sections = [s for s in sections if s in T.SECTIONS]
    reports = research_graph.invoke({"question": question, "sections": sections, "reports": []})["reports"]
    return json.dumps(reports), reports


SUPERVISOR_PROMPT = """You lead a small help-center team. You never answer the customer yourself.
Call the research tool once, with the customer's question word for word and every section it touches:
billing: plans, prices, payment methods, failed payments, invoices, currency, tax, renewing.
shipping: delivery times and costs, tracking, changing the address, territories, missing parcels.
returns: the return window, refunds, return shipping and labels, damaged items, items you can't return.
security: fraud, phishing, passwords, two-factor, charges you don't recognise, deleting an account.
If a part of the question still has no facts, call it again with other sections.
When every part has facts, or nothing more can be found, reply DONE."""

supervisor = create_agent(lab.model("supervisor"), tools=[research], system_prompt=SUPERVISOR_PROMPT,
                          middleware=[ModelCallLimitMiddleware(run_limit=4, exit_behavior="end")])

### 4c. One writer, and a checker with a checklist

**One writer.** Only the writer writes the answer, from the researchers' facts. When the
checker finds problems, the same writer fixes its own draft.

**The checker** starts fresh: it sees the question, the facts and the draft, and a
written checklist of pass or fail rules. It doesn't see how the draft was made. It
replies with a list of issues, empty when every check passes.

In [ ]:
WRITER_PROMPT = """Write the reply to a customer of an online shop, using only the facts below from its help center.
Use the facts' own words, and keep every number, timeframe, menu path and email address exactly as written.
Answer every part of the question, in under 80 words.
If the facts don't answer the question, reply exactly: {decline}

Question: {question}

Facts:
{facts}{fixes}"""

CHECKLIST = """You check a help-center reply before it goes to a customer. You did not write it.
Check it against this checklist, using only the facts below.
1. It answers every part of the question.
2. Every number, timeframe, menu path and email address in it appears in the facts.
3. It promises nothing the facts don't say.
4. If the facts don't answer the question, the reply is exactly: {decline}
Reply with JSON only: {{"issues": []}} if every check passes, otherwise list each problem, like
{{"issues": ["check 2: the facts don't say 10 days"]}}.

Question: {question}

Facts:
{facts}

Reply:
{draft}"""


def facts_text(reports):
    lines = [f"- {fact}" for report in reports for fact in report["facts"]]
    return "\n".join(lines) or "(none found)"


def writer(state):
    with mlflow.start_span(name="writer", span_type=SpanType.AGENT) as span:
        fixes = ""
        if state["issues"]:
            problems = "\n".join(f"- {issue}" for issue in state["issues"])
            fixes = f"\n\nYour last draft:\n{state['draft']}\n\nA checker found these problems. Fix them:\n{problems}"
        prompt = WRITER_PROMPT.format(decline=T.DECLINE, question=state["question"],
                                      facts=facts_text(state["reports"]), fixes=fixes)
        draft = lab.model("writer").invoke(prompt).content
        span.set_outputs({"draft": draft})
    return {"draft": draft}


def checker(state):
    with mlflow.start_span(name="checker", span_type=SpanType.AGENT) as span:
        prompt = CHECKLIST.format(decline=T.DECLINE, question=state["question"],
                                  facts=facts_text(state["reports"]), draft=state["draft"])
        issues = T.read_json(lab.model("checker").invoke(prompt).content, "issues")
        span.set_outputs({"issues": issues})
    return {"issues": issues, "rounds": state["rounds"] + 1}

**Your turn (exercise 1 of 5).** After each check, the team needs to know where to go
next. Write `after_check(issues, rounds, max_rounds=3)`:

- `issues` is the checker's list of problems, `rounds` is how many checks have run so far.
- No issues: return `"done"`.
- Issues, and `rounds` has reached `max_rounds`: return `"stop"`. The team hands the
  draft to a person instead of looping for ever.
- Otherwise return `"fix"`: back to the writer.

This is the lesson **A checker with a round limit**, as a LangGraph routing function.

In [ ]:
def after_check(issues, rounds, max_rounds=3):
    """Where the team goes after the checker: "done", "stop" or "fix"."""
    # TODO: no issues -> "done"; rounds at max_rounds or more -> "stop"; otherwise "fix".
    raise NotImplementedError("Milestone 10: implement after_check")

In [ ]:
%%ipytest -qq


def test_clean_draft_is_done():
    assert after_check([], 1) == "done"


def test_issues_go_back_to_the_writer():
    assert after_check(["check 1: misses the second part"], 1) == "fix"
    assert after_check(["check 2: 10 days is not in the facts"], 2) == "fix"


def test_the_third_round_is_the_last():
    assert after_check(["check 2: made-up email"], 3) == "stop"


def test_a_clean_last_round_still_counts_as_done():
    assert after_check([], 3) == "done"


def test_the_limit_can_change():
    assert after_check(["x"], 1, max_rounds=1) == "stop"

### 4d. Put the team together

The team is a LangGraph graph: supervisor, then writer, then checker, which loops back
to the writer until `after_check` says stop. At the round limit the draft goes to a
person (`flag`), and the plan counts that run as failed.

`build_team(with_checker=False)` gives the **ablation**: the same team with the checker
taken out, so section 5 can show what the checker buys.

In [ ]:
class Team(TypedDict):
    question: str
    reports: list
    draft: str
    issues: list
    rounds: int
    flagged: bool


def lead(state):
    with mlflow.start_span(name="supervisor", span_type=SpanType.AGENT) as span:
        span.set_inputs({"question": state["question"]})
        result = supervisor.invoke({"messages": [{"role": "user", "content": state["question"]}]})
        reports = []
        for message in result["messages"]:
            if message.type == "tool" and message.artifact:
                reports += message.artifact          # the researchers' reports, word for word
        span.set_outputs({"reports": reports})
    return {"reports": reports}


def flag(state):
    return {"flagged": True}


def build_team(with_checker=True):
    b = StateGraph(Team)
    b.add_node("supervisor", lead)
    b.add_node("writer", writer)
    b.add_edge(START, "supervisor")
    b.add_edge("supervisor", "writer")
    if with_checker:
        b.add_node("checker", checker)
        b.add_node("flag", flag)
        b.add_edge("writer", "checker")
        b.add_conditional_edges("checker", lambda s: after_check(s["issues"], s["rounds"]),
                                {"done": END, "stop": "flag", "fix": "writer"})
        b.add_edge("flag", END)
    else:
        b.add_edge("writer", END)
    return b.compile()


def runner(graph):
    def run(question):
        state = graph.invoke({"question": question, "reports": [], "draft": "", "issues": [],
                              "rounds": 0, "flagged": False})
        return {"answer": state["draft"], "stopped": state["flagged"]}
    return run


team = runner(build_team(with_checker=True))
team_no_checker = runner(build_team(with_checker=False))

with lab.request("demo", QUESTION["id"]):
    with mlflow.start_span(name="demo") as root:
        reply = team(QUESTION["question"])
mlflow.flush_trace_async_logging()
print(QUESTION["question"], "\n->", reply["answer"], "\n")
show_trace(root.trace_id)

Compare the two traces. The team's has one named box per helper (supervisor,
researcher, writer, checker), with its model calls inside. That shape, one workflow on
top and a named agent span per helper, is what **Reading agent traces** asks you to look
for, and it's what makes tokens per agent possible.

## 5. Run the experiment

### 5a. One more scorer: did the hand-off lose it?

`end_state` grades the outcome. For the teams, also check the **hand-off**: of the facts
the researchers found, how many reach the final answer? The supervisor can't reword
them (the artifact goes straight to the writer), but the writer still can.

**Your turn (exercise 2 of 5).** Write `handoff_kept(found, final)`, as in the lesson
**Did the hand-off lose it?**:

- `found`: the facts the researchers reported, like `["order 4412", "refund approved"]`.
- `final`: the final answer text.
- A fact is kept if it appears inside `final`, ignoring upper and lower case.
- Return `{"kept": share kept, rounded to 2 places, "lost": the facts not kept, sorted A to Z}`.
- No facts found: return `{"kept": 1.0, "lost": []}`.

In [ ]:
def handoff_kept(found, final):
    """The share of the helpers' facts that reach the final answer, and the ones that don't."""
    # TODO: lower-case the final text; keep each fact whose lower-case form is not in it in a list.
    raise NotImplementedError("Milestone 10: implement handoff_kept")

In [ ]:
%%ipytest -qq


def test_one_fact_lost():
    assert handoff_kept(["order 4412", "refund approved", "card ends 0099"],
                        "Order 4412: refund approved. It reaches you in 5 days.") == {"kept": 0.67, "lost": ["card ends 0099"]}


def test_case_does_not_matter():
    assert handoff_kept(["Express shipping costs $15"], "express shipping costs $15.") == {"kept": 1.0, "lost": []}


def test_nothing_found_means_nothing_lost():
    assert handoff_kept([], "Anything at all.") == {"kept": 1.0, "lost": []}


def test_lost_facts_are_sorted():
    assert handoff_kept(["ships monday", "order 77"], "Sorry.") == {"kept": 0.0, "lost": ["order 77", "ships monday"]}

The scorer reads the researchers' spans out of the trace with
`trace.search_spans(span_type=SpanType.AGENT, name="researcher")` and gives their facts
to your function.

In [ ]:
@scorer
def handoff(trace, outputs):
    found = []
    for span in trace.search_spans(span_type=SpanType.AGENT, name="researcher"):
        found += span.outputs["facts"]
    result = handoff_kept(found, outputs["answer"])
    return Feedback(value=result["kept"], rationale="lost: " + "; ".join(result["lost"]))

### 5b. Three designs, 3 runs each

Each design gets a name with `mlflow.set_active_model(name=...)`. Its `model_id` goes to
`mlflow.genai.evaluate`, so every trace knows which design made it. `evaluate` runs
each question once and has no repeat option, so it's called 3 times per design.

Each design's function is wrapped with `@mlflow.trace`, and tags its trace with the
question id, the run number and this session, so the traces can be lined up question
by question, and runs from an earlier session are left out.
Offline, this replays 180 runs in under a minute; live, it takes about 20 minutes.

In [ ]:
DESIGNS = {"one-agent": one_agent, "team": team, "team-no-checker": team_no_checker}


def traced(name, app):
    @mlflow.trace(name=name)
    def predict(qid, question):
        mlflow.update_current_trace(tags={"design": name, "qid": qid, "run": str(lab.run), "session": lab.session})
        with lab.request(name, qid):
            return app(question)
    return predict


MODEL_IDS = {}
for name, app in DESIGNS.items():
    model = mlflow.set_active_model(name=name)
    MODEL_IDS[name] = model.model_id
    scorers = [end_state] if name == "one-agent" else [end_state, handoff]
    for run in range(RUNS):
        lab.run = run
        mlflow.genai.evaluate(data=DATA, predict_fn=traced(name, app), scorers=scorers, model_id=model.model_id)
        print(f"{name}: run {run + 1} of {RUNS} done")
mlflow.flush_trace_async_logging()

## 6. Read the results

### 6a. Passed every time

Every trace now carries an `end_state` assessment. `mlflow.search_traces(model_id=...)`
fetches one design's traces, and `trace.search_assessments("end_state")` reads the
verdict. Grouped by question, that gives each question's tries:
`{"two-01": [True, True, False], ...}`.

**Your turn (exercise 3 of 5).** Write `every_time(runs)`: the share of questions
whose tries all passed. `all(list)` is True when every item in the list is True.

In [ ]:
def every_time(runs):
    """The share of questions passed on every try, like 0.75."""
    # TODO: count the questions whose tries are all True, then divide by the number of questions.
    raise NotImplementedError("Milestone 10: implement every_time")

In [ ]:
%%ipytest -qq


def test_one_failed_try_is_enough_to_miss():
    assert every_time({"q1": [True, True, True], "q2": [True, False, True]}) == 0.5


def test_all_passed():
    assert every_time({"q1": [True], "q2": [True, True]}) == 1.0


def test_none_passed_every_time():
    assert every_time({"q1": [False, True], "q2": [True, False]}) == 0.0


def test_four_questions():
    runs = {"a": [True] * 3, "b": [True] * 3, "c": [True] * 3, "d": [False, True, True]}
    assert every_time(runs) == 0.75

Now one row per design. **Dollars per success** is each trace's tokens
(`trace.info.token_usage`) at the example prices, added up over every run, then divided
by the runs that passed. **Model seconds** is the time the model took per run, from the
recording.

In [ ]:
def this_session(name):
    """One design's traces from this session (earlier sessions are left out)."""
    traces = mlflow.search_traces(model_id=MODEL_IDS[name], return_type="list")
    return [t for t in traces if t.info.tags.get("session") == lab.session]


TRACES = {name: this_session(name) for name in DESIGNS}


def passed(trace):
    found = trace.search_assessments("end_state")
    return bool(found) and found[0].value is True


def tries_of(traces):
    runs = {}
    for t in traces:
        runs.setdefault(t.info.tags["qid"], []).append(passed(t))
    return runs


def summary(name):
    traces = TRACES[name]
    runs = tries_of(traces)
    cost = sum(T.dollars(t.info.token_usage) for t in traces)
    wins = sum(passed(t) for t in traces)
    kept = [t.search_assessments("handoff")[0].value for t in traces if t.search_assessments("handoff")]
    seconds = [s for (design, run, qid), s in lab.seconds.items() if design == name]
    return {
        "design": name,
        "runs": len(traces),
        "pass rate": round(wins / len(traces), 2),
        "passed every time": round(every_time(runs), 2),
        "dollars": round(cost, 3),
        "dollars per success": round(cost / wins, 4) if wins else None,
        "model seconds per run": round(sum(seconds) / len(seconds), 1),
        "hand-off kept": round(sum(kept) / len(kept), 2) if kept else None,
    }


RESULTS = pd.DataFrame([summary(name) for name in DESIGNS]).set_index("design")
RESULTS

**Which questions changed?** With 20 questions, a gap of one or two is noise, so look at
the questions one by one. Each cell shows the 3 tries: `+` passed, `.` failed.

In [ ]:
def marks(tries):
    return "".join("+" if ok else "." for ok in tries)


TRIES = {name: tries_of(TRACES[name]) for name in DESIGNS}
by_question = pd.DataFrame({name: {q["id"]: marks(TRIES[name][q["id"]]) for q in T.QUESTIONS} for name in DESIGNS})
by_question.insert(0, "group", [q["group"] for q in T.QUESTIONS])
by_question

In [ ]:
one_ok = {qid for qid, tries in TRIES["one-agent"].items() if all(tries)}
team_ok = {qid for qid, tries in TRIES["team"].items() if all(tries)}
print("passed every time by the team only:     ", sorted(team_ok - one_ok))
print("passed every time by one agent only:    ", sorted(one_ok - team_ok))

group_rate = by_question.copy()
for name in DESIGNS:
    group_rate[name] = [all(TRIES[name][qid]) for qid in by_question.index]
print()
print(group_rate.groupby("group")[list(DESIGNS)].mean().round(2))

### 6b. Which helper costs the most?

Every model call's span holds its tokens, in the attribute `mlflow.chat.tokenUsage`
(`SpanAttributeKey.CHAT_USAGE`). Walk up from each model call to the nearest `AGENT`
span, and you know which helper made it. `calls_from_trace` does that walk.

In [ ]:
def calls_from_trace(trace):
    by_id = {s.span_id: s for s in trace.data.spans}
    calls = []
    for s in trace.data.spans:
        usage = s.get_attribute(SpanAttributeKey.CHAT_USAGE)
        if not usage:
            continue
        parent = by_id.get(s.parent_id)
        while parent is not None and parent.span_type != SpanType.AGENT:
            parent = by_id.get(parent.parent_id)
        calls.append({"agent": parent.name if parent else "unnamed",
                      "input_tokens": usage["input_tokens"], "output_tokens": usage["output_tokens"]})
    return calls


calls_from_trace(TRACES["team"][0])

**Your turn (exercise 4 of 5).** Write `tokens_by_agent(calls)`, as in the lesson **Which
helper costs the most?**: add up each agent's tokens (input plus output) over all its
calls, and return `[the agent with the most, {agent: total, ...}]`. If two agents tie,
pick the name that comes first A to Z.

In [ ]:
def tokens_by_agent(calls):
    """[the agent that used the most tokens, a dict of each agent's total tokens]."""
    # TODO: total each agent's input plus output tokens in a dict, then pick the biggest (A to Z on a tie).
    raise NotImplementedError("Milestone 10: implement tokens_by_agent")

In [ ]:
%%ipytest -qq


def test_two_agents():
    calls = [{"agent": "researcher", "input_tokens": 3000, "output_tokens": 400},
             {"agent": "writer", "input_tokens": 900, "output_tokens": 300},
             {"agent": "researcher", "input_tokens": 2500, "output_tokens": 500}]
    assert tokens_by_agent(calls) == ["researcher", {"researcher": 6400, "writer": 1200}]


def test_one_call():
    assert tokens_by_agent([{"agent": "agent", "input_tokens": 100, "output_tokens": 20}]) == ["agent", {"agent": 120}]


def test_a_tie_goes_to_the_first_name_a_to_z():
    calls = [{"agent": "writer", "input_tokens": 500, "output_tokens": 100},
             {"agent": "checker", "input_tokens": 400, "output_tokens": 200}]
    assert tokens_by_agent(calls) == ["checker", {"writer": 600, "checker": 600}]

In [ ]:
rows = {}
for name in DESIGNS:
    calls = [c for t in TRACES[name] for c in calls_from_trace(t)]
    top, totals = tokens_by_agent(calls)
    rows[name] = {agent: round(total / len(TRACES[name])) for agent, total in totals.items()}
    print(f"{name:16} biggest: {top}")

print("\nTokens per run, by agent:")
pd.DataFrame(rows).fillna(0).astype(int)

The biggest helper is where a cheaper model would save most. Try a smaller model for
that one helper only, and keep the change only if the eval still passes.

### 6c. Label the failures, then count

A pass rate says how often, not why. The lesson **Label failures, then count** says:
give each failed trace one label from a fixed list, count the labels, and fix the
biggest group first. The labels below are plain names for failure modes in the
**MAST** list.

A person should label failures. To start you off, `first_guess` labels each failed run
with simple rules, reading the trace: what the search or the researchers found, and
what the answer kept. Each label is saved on its trace with `mlflow.log_feedback`, so it
shows in the MLflow UI next to the run. Open a few and correct the ones the rules got
wrong: that's the real work.

In [ ]:
LABELS = {
    "answered what the help center doesn't say": "MAST: disobeyed the task",
    "found the facts, then left them out": "MAST: information withheld",
    "never found the facts": "a search problem, before any agent could fail",
    "made up a contact": "MAST: no or wrong checking",
    "stopped at a limit": "MAST: didn't know when to stop",
}


def what_was_found(trace):
    """Everything the search tool or the researchers handed back, as one text."""
    texts = [str(s.outputs) for s in trace.search_spans(span_type=SpanType.TOOL, name="search_help_center")]
    for s in trace.search_spans(span_type=SpanType.AGENT, name="researcher"):
        texts += s.outputs["facts"]
    return " ".join(texts)


def first_guess(trace):
    q = T.BY_ID[trace.info.tags["qid"]]
    root = next(s for s in trace.data.spans if s.parent_id is None)
    answer, stopped = root.outputs["answer"], root.outputs["stopped"]
    if stopped:
        return "stopped at a limit"
    if not q["facts"]:
        return "answered what the help center doesn't say"
    if T.made_up_contacts(answer):
        return "made up a contact"
    missing = [fact for fact in q["facts"] if not T.has_facts(answer, [fact])]
    if missing and all(T.has_facts(what_was_found(trace), [fact]) for fact in missing):
        return "found the facts, then left them out"
    return "never found the facts"


labelled = []
for name in DESIGNS:
    for t in TRACES[name]:
        if passed(t):
            continue
        label = first_guess(t)
        mlflow.log_feedback(trace_id=t.info.trace_id, name="failure", value=label,
                            source=AssessmentSource(source_type="CODE", source_id="first_guess"),
                            rationale=LABELS[label])
        labelled.append({"design": name, "label": label, "trace_id": t.info.trace_id, "qid": t.info.tags["qid"]})

LABELLED = pd.DataFrame(labelled)
pd.crosstab(LABELLED["label"], LABELLED["design"])

In [ ]:
# Two runs to read for each label, starting with the biggest group
for label in LABELLED["label"].value_counts().index:
    rows = LABELLED[LABELLED["label"] == label].head(2)
    print(f"{label}:")
    for row in rows.itertuples():
        print(f"   {row.design:16} {row.qid:8} {row.trace_id}")

## 7. Apply the rule, and only the rule

The rule from section 1: switch to the team only if its "passed every time" beats one
agent's by 0.15 or more, **and** its dollars per success is at most 0.05.

**Your turn (exercise 5 of 5).** Write `team_verdict(one, team, rule)`, as in the lesson
**One agent or a team: the experiment**:

- `one` and `team` look like `{"runs": {"two-01": [True, True, False], ...}, "cost": 0.12}`:
  each question's tries, and the dollars for all of them.
- `rule` is `{"min_gain": 0.15, "max_cost_per_success": 0.05}`.
- The gain is the team's `every_time` minus one agent's, **rounded to 2 places**.
- The team's dollars per success is its cost divided by its passed tries. With no passed
  tries, the team can't win.
- Return `"team"` if the gain is at least `min_gain` and the dollars per success is at
  most `max_cost_per_success`. Otherwise return `"one agent"`.

Use your `every_time` from exercise 3.

In [ ]:
def team_verdict(one, team, rule):
    """Apply the decision rule written before the runs: "team" or "one agent"."""
    # TODO: the gain in every_time (rounded to 2), the team's cost per passed try, then both checks.
    raise NotImplementedError("Milestone 10: implement team_verdict")

In [ ]:
%%ipytest -qq

RULE_FOR_TESTS = {"min_gain": 0.1, "max_cost_per_success": 0.5}


def test_team_wins_by_enough_at_a_fair_price():
    one = {"runs": {"a": [True, False], "b": [True, True]}, "cost": 0.2}
    team = {"runs": {"a": [True, True], "b": [True, True]}, "cost": 1.0}
    assert team_verdict(one, team, RULE_FOR_TESTS) == "team"


def test_gain_too_small():
    one = {"runs": {"a": [True], "b": [True], "c": [False]}, "cost": 0.1}
    team = {"runs": {"a": [True], "b": [True], "c": [False]}, "cost": 0.1}
    assert team_verdict(one, team, RULE_FOR_TESTS) == "one agent"


def test_wins_but_costs_too_much():
    one = {"runs": {"a": [False], "b": [True]}, "cost": 0.1}
    team = {"runs": {"a": [True], "b": [True]}, "cost": 1.2}
    assert team_verdict(one, team, RULE_FOR_TESTS) == "one agent"


def test_gain_of_exactly_the_margin_counts():
    # 10 questions: one agent passes 6 every time, the team 7, so the gain is exactly 0.1
    one = {"runs": {q: [q in "abcdef"] for q in "abcdefghij"}, "cost": 0.1}
    team = {"runs": {q: [q in "abcdefg"] for q in "abcdefghij"}, "cost": 0.1}
    assert team_verdict(one, team, RULE_FOR_TESTS) == "team"


def test_no_passes_cannot_win():
    one = {"runs": {"a": [False]}, "cost": 0.0}
    team = {"runs": {"a": [False]}, "cost": 0.3}
    assert team_verdict(one, team, RULE_FOR_TESTS) == "one agent"

In [ ]:
def side(name):
    return {"runs": TRIES[name], "cost": sum(T.dollars(t.info.token_usage) for t in TRACES[name])}


VERDICT = team_verdict(side("one-agent"), side("team"), RULE)
gain = round(every_time(TRIES["team"]) - every_time(TRIES["one-agent"]), 2)
print(f"passed every time: one agent {every_time(TRIES['one-agent']):.2f}, team {every_time(TRIES['team']):.2f}, gain {gain}")
print(f"team dollars per success: {RESULTS.loc['team', 'dollars per success']} (limit {RULE['max_cost_per_success']})")
print("verdict by the rule:", VERDICT)

**Read the ablation, too.** The decision is about one agent against the team. The
team without its checker explains the result: if it passes about as often as the
full team, the checker is only cost. If it passes clearly less, the checker is what
earns the team its place. Look back at the tables in 6a and the labels in 6c: the
"made up a contact" group is the one a checker is meant to catch.

**Before you trust the verdict,** look at the questions that changed (6a). If the
win rests on one or two questions, it's noise, and the plan says to keep one agent.

In [ ]:
for name in DESIGNS:
    row = RESULTS.loc[name]
    print(f"{name:16} passed every time {row['passed every time']:.2f}   pass rate {row['pass rate']:.2f}   "
          f"${row['dollars per success']} per success   {row['model seconds per run']} model seconds a run")

**What the recording shows.** On the recorded run, the rule says **keep one agent**: the
team passed every time on 9 questions out of 20 against one agent's 11, at about three
times the dollars per success and three times the model time. The team did better on
the questions the help center can't answer, and worse on the two-article questions it
was meant to help with.

The ablation explains a lot of it. Without its checker, the team made up a support
email address in 29 runs; with it, 10. So the checker does catch made-up contacts, and
lifts the team's pass rate from 0.47 to 0.65. But when the writer didn't fix the
problem, the checker sent the run to a person after 3 rounds (8 runs), and that still
counts as failed. The biggest failure group for every design is the same, made-up
contacts, so the cheapest next step is a better prompt for one agent, not a team.

If you ran it live, your numbers will differ. Apply the same rule to yours.

## 8. Browse the traces, and where this goes next

Open the traces to see what the tables summarise:

```bash
mlflow ui --backend-store-uri sqlite:///.drill/m10/mlflow.db --port 5052
```

Pick a model (one-agent, team or team-no-checker) to see its runs, filter by
`tags.qid = 'two-01'`, and open a trace: the named agent spans, each model call's tokens,
and the `end_state`, `handoff` and `failure` assessments are all there. (The test suite
writes to a temporary folder instead, so its runs don't show here.)

**Deploying on Databricks.** On Azure Databricks, the current way to ship a code-based
agent, one agent or a team, is a LangGraph app deployed with the **Agent Bricks CLI**,
with Genie Agents and other agents added as tools, like the researchers here. The
Databricks docs (October 2026) list **Supervisor Agent**, the **Supervisor API** and the
old "build a custom multi-agent system" guide as **legacy**, so don't start new work on
them. The same MLflow traces and scorers carry over: run this experiment on your
gateway's Azure AI Foundry models before you ship a team.